# Implicit Variable-Step Batched ODE Solver (JAX + Colab GPU)

This notebook provides a batched implicit variable-step solver:
- Backward Euler startup
- Variable-step BDF2 progression
- Newton iterations for implicit solves

Designed for Google Colab GPU execution with JAX.


In [ ]:
# Colab GPU setup (run once, then Runtime -> Restart runtime)
# If JAX is already GPU-enabled, you can skip this.
%pip -q install -U "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


In [ ]:
import time
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# float64 required: newton_tol=1e-10 and rtol=1e-6 both demand full double precision.
jax.config.update("jax_enable_x64", True)

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())
print('Default backend:', jax.default_backend())

In [ ]:
# ---------------------------------------------------------------------------
# Implicit variable-step BDF2 (BE startup + adaptive BDF2) — JAX/GPU solver
# ---------------------------------------------------------------------------

def rhs_impvar_single(t, y, lam):
    """Stiff test: y' = -lam*(y - cos(t)) - sin(t).
    Exact solution: y(t) = (y0 - 1)*exp(-lam*t) + cos(t).
    """
    return -lam * (y - jnp.cos(t)) - jnp.sin(t)


def jac_impvar_single(t, y, lam):
    """Analytic diagonal Jacobian."""
    return -jnp.diag(lam)


def error_norm_batch_iv(y_old, y_new, err, rtol, atol):
    scale = atol + rtol * jnp.maximum(jnp.abs(y_old), jnp.abs(y_new))
    ratio = err / scale
    return jnp.sqrt(jnp.mean(ratio * ratio, axis=1))


def newton_be_batch_iv(y_prev, t_next, h, lam, tol=1e-10, max_iter=20):
    """Batched Newton solve for BE: y - y_prev - h*f(t_next, y) = 0."""
    batch, dim = y_prev.shape
    eye = jnp.eye(dim)
    y = y_prev.copy()
    converged = jnp.zeros((batch,), dtype=bool)
    iters     = jnp.zeros((batch,), dtype=jnp.int32)

    for k in range(max_iter):
        f     = jax.vmap(rhs_impvar_single)(t_next, y, lam)
        res   = y - y_prev - h[:, None] * f
        res_n = jnp.max(jnp.abs(res), axis=1)

        just_conv = (~converged) & (res_n < tol)
        iters     = jnp.where(just_conv, k + 1, iters)
        converged = converged | just_conv
        if bool(jnp.all(converged)):
            break

        Jf    = jax.vmap(jac_impvar_single)(t_next, y, lam)
        A     = eye[None] - h[:, None, None] * Jf
        delta = jnp.linalg.solve(A, -res[..., None]).squeeze(-1)
        delta = jnp.where(converged[:, None], 0.0, delta)
        y     = y + delta

        delta_n   = jnp.max(jnp.abs(delta), axis=1)
        just_conv2 = (~converged) & (delta_n < tol)
        iters      = jnp.where(just_conv2, k + 2, iters)
        converged  = converged | just_conv2

    iters = jnp.where((iters == 0) & converged, 1, iters)
    iters = jnp.where(~converged, max_iter, iters)
    return y, converged, iters


def newton_bdf2_batch_iv(y_nm1, y_n, t_next, h_prev, h, lam, tol=1e-10, max_iter=20):
    """Batched Newton solve for variable-step BDF2.

    Coefficients (r = h/h_prev, uniform step r=1 → standard BDF2):
        a0 = (1+2r)/(h*(1+r))
        a1 = -(1+r)/h
        a2 =  r^2/(h*(1+r))
    Residual: a0*y + a1*y_n + a2*y_nm1 - f(t_next, y) = 0
    """
    batch, dim = y_n.shape
    eye = jnp.eye(dim)

    r  = h / h_prev
    a0 = (1.0 + 2.0 * r) / (h * (1.0 + r))
    a1 = -(1.0 + r) / h
    a2 = (r * r) / (h * (1.0 + r))

    # Linear extrapolation predictor.
    y  = y_n + (h / h_prev)[:, None] * (y_n - y_nm1)
    converged = jnp.zeros((batch,), dtype=bool)
    iters     = jnp.zeros((batch,), dtype=jnp.int32)

    for k in range(max_iter):
        f     = jax.vmap(rhs_impvar_single)(t_next, y, lam)
        res   = a0[:, None]*y + a1[:, None]*y_n + a2[:, None]*y_nm1 - f
        res_n = jnp.max(jnp.abs(res), axis=1)

        just_conv = (~converged) & (res_n < tol)
        iters     = jnp.where(just_conv, k + 1, iters)
        converged = converged | just_conv
        if bool(jnp.all(converged)):
            break

        Jf    = jax.vmap(jac_impvar_single)(t_next, y, lam)
        A     = a0[:, None, None] * eye[None] - Jf
        delta = jnp.linalg.solve(A, -res[..., None]).squeeze(-1)
        delta = jnp.where(converged[:, None], 0.0, delta)
        y     = y + delta

        delta_n   = jnp.max(jnp.abs(delta), axis=1)
        just_conv2 = (~converged) & (delta_n < tol)
        iters      = jnp.where(just_conv2, k + 2, iters)
        converged  = converged | just_conv2

    iters = jnp.where((iters == 0) & converged, 1, iters)
    iters = jnp.where(~converged, max_iter, iters)
    return y, converged, iters


def solve_implicit_variable_batch_jax(
    y0_batch,
    lam_batch,
    t_span=(0.0, 6.0),
    rtol=1e-6,
    atol=1e-8,
    h_init=1e-3,
    h_min=1e-7,
    h_max=0.2,
    max_iters=200_000,
    newton_tol=1e-10,
    newton_max_iter=20,
    safety=0.9,
    min_factor=0.2,
    max_factor=3.0,
    track_index=0,
):
    """Batched adaptive BDF2 solver (BE startup + variable-step BDF2).

    Algorithm:
      - Step 1 accepted unconditionally using Backward Euler (startup).
      - Subsequent steps: compute both BE and BDF2; use (BDF2 - BE) as an
        O(h^2) error estimate; control step via err^{-1/(p+1)}, p=2.
    """
    t0, tf = float(t_span[0]), float(t_span[1])
    y_n  = jnp.array(y0_batch, dtype=jnp.float64)
    lam  = jnp.array(lam_batch, dtype=jnp.float64)

    batch   = y_n.shape[0]
    t       = jnp.full((batch,), t0)
    h       = jnp.full((batch,), h_init)
    h_prev  = jnp.full((batch,), h_init)
    y_nm1   = y_n
    has_prev = jnp.zeros((batch,), dtype=bool)
    done    = jnp.zeros((batch,), dtype=bool)

    n_accept = jnp.zeros((batch,), dtype=jnp.int32)
    n_reject = jnp.zeros((batch,), dtype=jnp.int32)
    n_newton = jnp.zeros((batch,), dtype=jnp.int32)

    t_hist = [float(t[track_index])]
    y_hist = [np.array(y_n[track_index])]

    for _ in range(max_iters):
        active = ~done
        if not bool(jnp.any(active)):
            break

        h_eff  = jnp.where(active, jnp.minimum(h, tf - t), h)
        h_eff  = jnp.clip(h_eff, h_min, h_max)
        t_next = t + h_eff

        # --- Backward Euler (always needed for error estimate) ---
        y_be, ok_be, nit_be = newton_be_batch_iv(
            y_prev=y_n, t_next=t_next, h=h_eff, lam=lam,
            tol=newton_tol, max_iter=newton_max_iter,
        )
        n_newton = n_newton + nit_be

        # --- BDF2 (only when previous step exists) ---
        h_prev_safe = jnp.where(h_prev > 0, h_prev, h_init)
        y_bdf2, ok_bdf2, nit_bdf2 = newton_bdf2_batch_iv(
            y_nm1=y_nm1, y_n=y_n, t_next=t_next,
            h_prev=h_prev_safe, h=h_eff, lam=lam,
            tol=newton_tol, max_iter=newton_max_iter,
        )
        n_newton = n_newton + jnp.where(has_prev, nit_bdf2, 0)

        # --- Error estimate: BDF2 - BE (O(h^2) difference) ---
        err_bdf2 = error_norm_batch_iv(y_n, y_bdf2, y_bdf2 - y_be, rtol=rtol, atol=atol)
        err      = jnp.where(has_prev, err_bdf2, 0.0)  # startup: always accept

        can_accept = ok_be & (~has_prev | ok_bdf2)
        accept     = active & can_accept & (err <= 1.0)

        y_high    = jnp.where(has_prev[:, None], y_bdf2, y_be)
        y_nm1_new = jnp.where(accept[:, None], y_n, y_nm1)
        y_n_new   = jnp.where(accept[:, None], y_high, y_n)
        t_new     = jnp.where(accept, t_next, t)
        h_prev_new = jnp.where(accept, h_eff, h_prev)
        has_prev_new = has_prev | accept

        # --- Step-size control ---
        p        = jnp.where(has_prev, 2.0, 1.0)
        safe_err = jnp.where(err > 0, err, 1e-300)
        f_accept = jnp.where(
            err == 0.0, max_factor,
            jnp.clip(safety * safe_err**(-1.0/(p+1.0)), min_factor, max_factor),
        )
        f_reject = jnp.where(
            has_prev,
            jnp.clip(safety * safe_err**(-1.0/(p+1.0)), min_factor, 1.0),
            0.5,
        )
        h_prop   = jnp.where(accept, h_eff * f_accept, h_eff * f_reject)
        h_new    = jnp.where(active, jnp.clip(h_prop, h_min, h_max), h)
        done_new = done | (t_new >= (tf - 1e-12))

        n_accept = n_accept + accept.astype(jnp.int32)
        n_reject = n_reject + (~accept & active).astype(jnp.int32)

        y_nm1, y_n, t, h_prev, has_prev, h, done = (
            y_nm1_new, y_n_new, t_new, h_prev_new, has_prev_new, h_new, done_new,
        )

        t_hist.append(float(t[track_index]))
        y_hist.append(np.array(y_n[track_index]))

    return {
        't_final':        np.array(t),
        'y_final':        np.array(y_n),
        'n_accept':       np.array(n_accept),
        'n_reject':       np.array(n_reject),
        'n_newton_total': np.array(n_newton),
        't_hist':         np.array(t_hist),
        'y_hist':         np.array(y_hist),
    }


def exact_impvar(t, y0, lam):
    """Exact solution: y(t) = (y0 - 1)*exp(-lam*t) + cos(t)."""
    return (y0 - 1.0) * np.exp(-lam * t) + np.cos(t)

In [ ]:
rng = np.random.default_rng(2)
batch_size = 512
dim = 4
t_span = (0.0, 6.0)
tf = t_span[1]

y0_batch  = rng.normal(size=(batch_size, dim))
lam_batch = rng.uniform(5.0, 60.0, size=(batch_size, dim))

cfg = dict(
    t_span=t_span, rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_max=0.1,
    newton_tol=1e-10, newton_max_iter=20,
    track_index=0,
)

# --- Warmup ---
print("Warming up JAX...")
_ = solve_implicit_variable_batch_jax(y0_batch[:4], lam_batch[:4], max_iters=20, **cfg)

# --- Timed run ---
start = time.perf_counter()
out = solve_implicit_variable_batch_jax(y0_batch, lam_batch, **cfg)
elapsed = time.perf_counter() - start

print(f"\nImplicit variable-step BDF2 (JAX/GPU)  batch={batch_size}  dim={dim}")
print(f"  Runtime           : {elapsed:.3f} s")
print(f"  Throughput        : {batch_size/elapsed:.0f} trajectories/s")
print(f"  Mean accepted     : {out['n_accept'].mean():.1f} steps")
print(f"  Mean rejected     : {out['n_reject'].mean():.1f} steps")
print(f"  Mean Newton total : {out['n_newton_total'].mean():.1f}")
mean_steps = out['n_accept'].mean() + out['n_reject'].mean()
print(f"  Mean Newton/step  : {out['n_newton_total'].mean() / max(1, mean_steps):.2f}")

# --- Correctness vs exact solution ---
errors = []
for i in range(batch_size):
    y_ex = exact_impvar(tf, y0_batch[i], lam_batch[i])
    errors.append(np.max(np.abs(out['y_final'][i] - y_ex)))
print(f"\n  Max |error| vs exact : {np.max(errors):.2e}  (rtol=1e-6, atol=1e-8)")
print(f"  Mean|error| vs exact : {np.mean(errors):.2e}")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(out['t_hist'], out['y_hist'][:, 0])
axes[0].set_xlabel('t'); axes[0].set_ylabel('y[0]')
axes[0].set_title('Tracked trajectory (implicit variable-step BDF2, JAX)')
axes[0].grid(True)

axes[1].semilogy(sorted(errors))
axes[1].axhline(1e-6, ls='--', color='r', label='rtol')
axes[1].set_xlabel('Trajectory index (sorted)')
axes[1].set_ylabel('|error| at t=tf')
axes[1].set_title('Per-trajectory error vs exact')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()